In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content')

!rm -rf /content/graduation-p
!git clone https://github.com/berk0vic/graduation-p.git /content/graduation-p

os.chdir('/content/graduation-p/notebooks')
sys.path.insert(0, '/content/graduation-p')

!rm -rf /content/graduation-p/data
!ln -s /content/drive/MyDrive/graduation-p/data /content/graduation-p/data

import torch
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html
!pip install -q xgboost shap pyyaml

# 05 — Explainability & Scalability

**Part A:** SHAP analysis — which embedding dimensions drive the fraud decision (IEEE-CIS)

**Part B:** Scalability test on PaySim (6M+ transactions) — graph construction and inference time

In [16]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

Device: cuda


---
## Part A: Explainability (SHAP)

Load the trained Hybrid GAT+VAE model and apply SHAP to the highest-scoring fraud transactions.

In [ ]:
# Load graph and model
from src.models.hybrid_model import HybridGATVAE

graph_path = '../data/processed/ieee_cis/hetero_graph_v3.pt'

if os.path.exists(graph_path):
    data = torch.load(graph_path, map_location='cpu', weights_only=False)
    print(f'Loaded pre-built graph from {graph_path}')
else:
    # Build graph from raw CSV
    from src.data.ieee_cis_loader import load_raw
    from src.graph.builder import build_hetero_graph
    print('Pre-built graph not found, building from CSV...')
    df = load_raw()
    data = build_hetero_graph(df, dataset='ieee_cis')
    print(f'Graph built: {data}')

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

in_channels = {ntype: data[ntype].x.shape[1] for ntype in data.node_types}
raw_txn_dim = data['transaction'].x.shape[1]

model = HybridGATVAE(
    metadata=data.metadata(),
    in_channels=in_channels,
    raw_txn_dim=raw_txn_dim,
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
)

weights_path = '../results/models/hybrid_gatvae_ieee_cis.pt'
if os.path.exists(weights_path):
    try:
        model.load_state_dict(torch.load(weights_path, map_location='cpu', weights_only=True))
        print('Model weights loaded.')
    except Exception:
        print('WARNING: Checkpoint mismatch, using random weights (re-train to fix)')
else:
    print('WARNING: No weights found, using random weights')

model.eval()
print(f'Node types: {data.node_types}')
print(f'Transaction features: {raw_txn_dim}')

In [ ]:
# Run inference to get GAT embeddings and fraud scores
with torch.no_grad():
    x_dict = {ntype: data[ntype].x for ntype in data.node_types}
    edge_dict = {etype: data[etype].edge_index for etype in data.edge_types}
    raw_txn = data['transaction'].x
    outputs = model(x_dict, edge_dict, raw_txn_features=raw_txn)

h_t = outputs['h_t'].numpy()
fraud_scores = outputs['fraud_score'].numpy()

print(f'Embedding shape: {h_t.shape}')
print(f'Fraud score range: [{fraud_scores.min():.4f}, {fraud_scores.max():.4f}]')

In [ ]:
import shap

# SHAP: top 10 fraud + 50 random as background
top_fraud_idx = np.argsort(fraud_scores)[-10:]
bg_idx = np.random.RandomState(42).choice(len(h_t), size=50, replace=False)

background = h_t[bg_idx]
explain_samples = h_t[top_fraud_idx]

# Predict function: embedding -> fraud score via classifier head
# We fix recon_err to zero so SHAP only explains the GAT embedding dimensions
def predict_fn(h_t_np):
    h_t_tensor = torch.tensor(h_t_np, dtype=torch.float32)
    with torch.no_grad():
        recon_err = torch.zeros(len(h_t_tensor), 1)
        cls_input = torch.cat([h_t_tensor, recon_err], dim=1)
        logit = model.classifier(cls_input).squeeze(-1)
        return torch.sigmoid(logit).numpy()

explainer = shap.KernelExplainer(predict_fn, background)
shap_values = explainer.shap_values(explain_samples, nsamples=200)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# SHAP bar plot: ortalama feature önemleri
feature_names = [f'emb_{i}' for i in range(h_t.shape[1])]

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_k = 15
top_features = np.argsort(mean_abs_shap)[-top_k:]

plt.figure(figsize=(10, 6))
plt.barh(
    [feature_names[i] for i in top_features],
    mean_abs_shap[top_features],
    color='#E74C3C'
)
plt.xlabel('Mean |SHAP value|')
plt.title('Top 15 Most Important Embedding Dimensions for Fraud Detection')
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/shap_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Tek bir fraud işlem için waterfall plot
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        feature_names=feature_names,
    ),
    max_display=15,
)

---
## Part B: Scalability (PaySim)

Testing on the 6.3M-transaction PaySim dataset:
1. Graph construction time at different scales
2. Inference latency (target: < 1 second for real-time alerting)

In [ ]:
from src.data.paysim_loader import load_raw, filter_fraud_types

print('Loading PaySim...')
t0 = time.time()
df_paysim = load_raw()
load_time = time.time() - t0

print(f'Load time: {load_time:.1f}s')
print(f'Total transactions: {len(df_paysim):,}')
print(f'Fraud: {df_paysim["isFraud"].sum():,} ({100*df_paysim["isFraud"].mean():.3f}%)')

# Filter to TRANSFER + CASH_OUT (where fraud happens)
df_filtered = filter_fraud_types(df_paysim)
print(f'\nFiltered (TRANSFER+CASH_OUT): {len(df_filtered):,}')
print(f'Fraud: {df_filtered["isFraud"].sum():,} ({100*df_filtered["isFraud"].mean():.3f}%)')

In [ ]:
# Graph construction time at different scales
from src.graph.builder import build_hetero_graph

sample_sizes = [10_000, 50_000, 100_000, 500_000]
graph_times = []

for n in sample_sizes:
    df_sample = df_filtered.head(n)
    t0 = time.time()
    g = build_hetero_graph(df_sample, dataset='paysim')
    elapsed = time.time() - t0
    graph_times.append(elapsed)
    n_nodes = sum(g[nt].x.shape[0] for nt in g.node_types)
    n_edges = sum(g[et].edge_index.shape[1] for et in g.edge_types)
    print(f'{n:>10,} txns → {n_nodes:>10,} nodes, {n_edges:>10,} edges | {elapsed:.2f}s')

In [ ]:
# Inference latency test on 100K PaySim graph
test_graph = build_hetero_graph(df_filtered.head(100_000), dataset='paysim')
test_graph['transaction'].y = torch.zeros(test_graph['transaction'].x.shape[0], dtype=torch.long)

in_ch = {ntype: test_graph[ntype].x.shape[1] for ntype in test_graph.node_types}
raw_dim = test_graph['transaction'].x.shape[1]

test_model = HybridGATVAE(
    metadata=test_graph.metadata(),
    in_channels=in_ch,
    raw_txn_dim=raw_dim,
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
)
test_model.eval()

x_dict = {ntype: test_graph[ntype].x for ntype in test_graph.node_types}
edge_dict = {etype: test_graph[etype].edge_index for etype in test_graph.edge_types}
raw_txn = test_graph['transaction'].x

# Warmup
with torch.no_grad():
    _ = test_model(x_dict, edge_dict, raw_txn_features=raw_txn)

# Measure average over 5 runs
latencies = []
for _ in range(5):
    t0 = time.time()
    with torch.no_grad():
        out = test_model(x_dict, edge_dict, raw_txn_features=raw_txn)
    latencies.append(time.time() - t0)

avg_latency = np.mean(latencies)
print(f'Inference latency (100K transactions):')
print(f'  Average: {avg_latency:.3f}s')
print(f'  Per-transaction: {1000*avg_latency/100_000:.4f}ms')
print(f'  Target (< 1s): {"PASS" if avg_latency < 1.0 else "EXCEEDED"}')

In [ ]:
# Scalability charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graph construction time
axes[0].plot(sample_sizes, graph_times, 'o-', color='#3498DB', linewidth=2)
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Graph Construction Time')
axes[0].grid(True, alpha=0.3)

# Inference latency
axes[1].bar(['100K\nTransactions'], [avg_latency], color='#E74C3C', width=0.4)
axes[1].axhline(y=1.0, color='green', linestyle='--', label='1s target')
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Inference Latency')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/scalability.png', dpi=150)
plt.show()

In [ ]:
# Save scalability results
scalability_results = {
    'graph_construction': {
        'sample_sizes': sample_sizes,
        'times_seconds': graph_times,
    },
    'inference_latency': {
        'num_transactions': 100_000,
        'avg_seconds': avg_latency,
        'per_transaction_ms': 1000 * avg_latency / 100_000,
        'under_1s': avg_latency < 1.0,
    },
}

os.makedirs('../results', exist_ok=True)
with open('../results/scalability_results.json', 'w') as f:
    json.dump(scalability_results, f, indent=2)

print('Results saved: results/scalability_results.json')